### Note on outputs in this notebook

This notebook has been switched over from **OpenAI** to **Gemini** (via the native
`google-genai` SDK) for embeddings + generation, and adds a **Groq**-backed node
for fast intent classification, using the client setup you provided:

```python
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in .env file!")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env file!")

gemini_client = genai.Client(api_key=GOOGLE_API_KEY)
ls_client = Client()
```

> **Important:** the printed cell outputs below are **illustrative / simulated**,
> not the result of a live run. This sandbox has no network access to
> `generativelanguage.googleapis.com`, `api.groq.com`, or a running Qdrant
> instance, so the notebook could not actually be executed here. Once you drop
> in your `.env` (with `GOOGLE_API_KEY` and `GROQ_API_KEY`) and have Qdrant
> running locally with a Gemini-embedding collection, running the cells
> top-to-bottom will produce real output in place of these.

### Import Dependencies

In [ ]:
from pydantic import BaseModel, Field

from qdrant_client import QdrantClient
from qdrant_client.models import Prefetch, Filter, FieldCondition, MatchText, FusionQuery, Document

from langsmith import traceable, get_current_run_tree
from langsmith import Client

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.types import Send, Command

from langchain_core.messages import AIMessage, ToolMessage

from jinja2 import Template
from typing import Literal, Dict, Any, Annotated, List, Optional, Sequence
from IPython.display import Image, display
from operator import add

import os
import random
import ast
import inspect
import instructor
import json

# --- Gemini (native google-genai SDK) ---
from google import genai
from google.genai import types

# --- Groq (used for a fast/cheap intent-classification node) ---
from groq import Groq

from dotenv import load_dotenv

from notebooks.week_3.utils.utils_1 import get_tool_descriptions, format_ai_message

In [1]:
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in .env file!")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env file!")

# Native google-genai client -> used for the RAG pipeline itself
# (embeddings + answer generation)
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

# Native Groq client -> used for the lightweight intent-routing node
groq_client = Groq(api_key=GROQ_API_KEY)

# LangSmith client -> used to pull the evaluation dataset
ls_client = Client()

# instructor-patched clients: these wrap the native clients above so we can
# request structured (Pydantic) outputs from each provider.
gemini_structured_client = instructor.from_genai(gemini_client)
groq_structured_client = instructor.from_groq(groq_client, mode=instructor.Mode.TOOLS)

GEMINI_MODEL = "gemini-2.5-flash"
GEMINI_EMBEDDING_MODEL = "gemini-embedding-001"
GROQ_MODEL = "llama-3.3-70b-versatile"

print("Gemini client ready:", gemini_client is not None)
print("Groq client ready:", groq_client is not None)
print("LangSmith client ready:", ls_client is not None)

Gemini client ready: True
Groq client ready: True
LangSmith client ready: True


#### Query Expansion (Parallel Execution)

In [ ]:
class State(BaseModel):
    expanded_query: List[str] = []
    retrieved_context: Annotated[List[str], add] = []
    question_relevant: bool = False
    initial_query: str = ""
    answer: str = ""
    query: str = ""
    k: int = 10

#### Query Expansion / Rewriting Node

In [ ]:
class QueryExpandResponse(BaseModel):
   expanded_query: List[str]

In [ ]:
@traceable(
    name="query_expand_node",
    run_type="llm",
    metadata={"ls_provider": "google_genai", "ls_model_name": GEMINI_MODEL}
)
def query_expand_node(state: State) -> dict:

   prompt_template =  """You are part of a shopping assistant that can answer questions about products in stock.

Instructions:
- You will be given a question and you need to expand it into a list of statements that can be used in contextual search to retrieve relevant products.
- The statements should not overlap in context.
- The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

<Question>
{{ query }}
</Question>
"""

   template = Template(prompt_template)

   prompt = template.render(
      query=state.initial_query
   )

   response, raw_response = gemini_structured_client.create_with_completion(
        model=GEMINI_MODEL,
        response_model=QueryExpandResponse,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
   )

   return {
      "expanded_query": response.expanded_query
   }

In [ ]:
def query_expand_conditional_edges(state: State):

    send_messages = []

    for query in state.expanded_query:
        send_messages.append(
            Send(
                "retrieve_node",
                {
                    "query": query,
                    "k": 10
                }
            )
        )

    return send_messages

#### Retriever Node

In [ ]:
@traceable(
    name="embed_query",
    run_type="embedding",
    metadata={"ls_provider": "google_genai", "ls_model_name": GEMINI_EMBEDDING_MODEL}
)
def get_embedding(text, model=GEMINI_EMBEDDING_MODEL):
    response = gemini_client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    )
    return response.embeddings[0].values


@traceable(
    name="retrieve_top_n",
    run_type="retriever"
)
def retrieve_node(state: State) -> dict:

    qdrant_client = QdrantClient(url="http://localhost:6333")

    query_embedding = get_embedding(state["query"])

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search-gemini",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="gemini-embedding-001",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=state["query"],
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=state["k"],
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        retrieved_context_ratings.append(result.payload["average_rating"])
        similarity_scores.append(result.score)

    formatted_context = ""

    for id, chunk, rating in zip(retrieved_context_ids, retrieved_context, retrieved_context_ratings):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return {
        "retrieved_context": [formatted_context]
    }

#### Aggregator Node

In [ ]:
class AggregatorResponse(BaseModel):
    answer: str = Field(description="Answer to the question.")

In [ ]:
@traceable(
    name="aggregator_node",
    run_type="llm",
    metadata={"ls_provider": "google_genai", "ls_model_name": GEMINI_MODEL}
)
def aggregator_node(state: State) -> dict:

   preprocessed_context = "\n".join(state.retrieved_context)

   prompt_template =  """You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

Context:
{{ preprocessed_context }}

Question:
{{ question }}
"""

   template = Template(prompt_template)

   prompt = template.render(
      preprocessed_context=preprocessed_context,
      question=state.initial_query
   )

   response, raw_response = gemini_structured_client.create_with_completion(
        model=GEMINI_MODEL,
        response_model=AggregatorResponse,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
   )

   return {
      "answer": response.answer
   }

#### User Intent Router Node\n\nThis node now runs on **Groq** (e.g. `llama-3.3-70b-versatile`) instead of OpenAI, since intent classification is a small/cheap task where Groq's low-latency inference is a good fit, while the heavier generation/embedding work stays on Gemini.

In [ ]:
class IntentRouterResponse(BaseModel):
    question_relevant: bool
    answer: str

In [ ]:
@traceable(
    name="agent_node",
    run_type="llm",
    metadata={"ls_provider": "groq", "ls_model_name": GROQ_MODEL}
)
def intent_router_node(state: State):

   prompt_template =  """You are part of a shopping assistant that can answer questions about products in stock.

Instructions:
- You will be given a question and you need to clasify it into relevant or not relevant.
- If the question is not relevant, return False in field "question_relevant" and set "answer" to explanation why it is not relevant.
- If the question is relevant, return True in field "question_relevant" and set "answer" to "".
- You should only answer questions about the products in stock. If the question is not about the products in stock, you should ask for clarification.

<Question>
{{ query }}
</Question>
"""

   template = Template(prompt_template)

   prompt = template.render(
      query=state.initial_query
   )

   response, raw_response = groq_structured_client.create_with_completion(
        model=GROQ_MODEL,
        response_model=IntentRouterResponse,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
   )

   return {
      "question_relevant": response.question_relevant,
      "answer": response.answer
      }

In [ ]:
def intent_router_conditional_edges(state: State):

    if state.question_relevant:
        return "query_expand_node"
    else:
        return "end"

#### Graph

In [1]:
workflow = StateGraph(State)

workflow.add_node("query_expand_node", query_expand_node)
workflow.add_node("retrieve_node", retrieve_node)
workflow.add_node("aggregator_node", aggregator_node)
workflow.add_node("intent_router_node", intent_router_node)

workflow.add_edge(START, "intent_router_node")
workflow.add_conditional_edges("query_expand_node", query_expand_conditional_edges)
workflow.add_conditional_edges(
    "intent_router_node",
    intent_router_conditional_edges,
    {
        "query_expand_node": "query_expand_node",
        "end": END
    }
)

workflow.add_edge("retrieve_node", "aggregator_node")
workflow.add_edge("aggregator_node", END)

graph = workflow.compile()
print("Graph compiled with nodes:", list(graph.get_graph().nodes.keys()))

Graph compiled with nodes: ['__start__', 'query_expand_node', 'retrieve_node', 'aggregator_node', 'intent_router_node', '__end__']


In [1]:
display(Image(graph.get_graph().draw_mermaid_png()))

[Mermaid graph image rendered here]

In [1]:
initial_state = {
    "initial_query": "Can I get a tablet for my kid, a watch for me a laptop for my wife and a waterproof speaker for our party next week?"
}
result = graph.invoke(initial_state)

In [1]:
result

{'expanded_query': ['Kid-friendly tablet with parental controls and durable case',
 "Men's smartwatch with fitness tracking and long battery life",
 "Lightweight laptop for everyday productivity, suitable as a gift for a wife",
 'Waterproof portable Bluetooth speaker for outdoor parties'],
 'retrieved_context': ['- ID: B0C1H2X3Y4, rating: 4.5, description: 10-inch kids tablet with parental controls, 32GB storage, protective bumper case...\n- ID: B0D5F6G7H8, rating: 4.3, description: Android smartwatch with heart-rate and sleep tracking, 5-day battery life, GPS...\n',
 '- ID: B0E9J1K2L3, rating: 4.6, description: 14-inch ultra-light laptop, Intel Core i5, 16GB RAM, 512GB SSD, all-day battery...\n- ID: B0F4M5N6P7, rating: 4.4, description: IPX7 waterproof portable Bluetooth speaker, 20-hour playtime, built-in mic...\n'],
 'question_relevant': True,
 'initial_query': 'Can I get a tablet for my kid, a watch for me a laptop for my wife and a waterproof speaker for our party next week?',
 'a

In [1]:
print(result["answer"])

Here are some great options from our available products:

**Kids Tablet**
- 10-inch display with parental controls
- 32GB storage, protective bumper case
- Rating: 4.5

**Smartwatch**
- Heart-rate and sleep tracking
- 5-day battery life, built-in GPS
- Rating: 4.3

**Laptop**
- 14-inch, Intel Core i5, 16GB RAM, 512GB SSD
- All-day battery life
- Rating: 4.6

**Waterproof Speaker**
- IPX7 waterproof rating
- 20-hour playtime, built-in mic
- Rating: 4.4


In [1]:
initial_state = {
    "initial_query": "Whats the weather today?"
}
result = graph.invoke(initial_state)

In [1]:
result

{'expanded_query': [],
 'retrieved_context': [],
 'question_relevant': False,
 'initial_query': 'Whats the weather today?',
 'answer': "I'm a shopping assistant and can only help with questions about the products we have in stock, such as electronics, accessories, or gifts. I'm not able to provide weather information -- would you like help finding a product instead?",
 'query': '',
 'k': 10}